In [20]:
import pandas as pd
import numpy as np
from scipy import stats

df = pd.read_excel(r'C:\Users\slava\OneDrive\Рабочий стол\PyDissertation\survey.xlsx')  

# Column indices
awareness_cols = list(df.columns[[5, 6, 8, 9]])  # Q1,Q2,Q4,Q5 (Q3 excluded)
trust_cols     = list(df.columns[10:15])           # Block 3: Q1-Q5
creepy_cols    = list(df.columns[15:20])           # Block 4: Q1-Q5
 
# Convert to numeric
for cols in [awareness_cols, trust_cols, creepy_cols]:
    df[cols] = df[cols].apply(pd.to_numeric, errors='coerce')

In [21]:
# ================================================================
# CRONBACH'S ALPHA
# ================================================================
def cronbach_alpha(data):
    data = np.array(data, dtype=float)
    data = data[~np.isnan(data).any(axis=1)]
    k = data.shape[1]
    n = data.shape[0]
    item_vars = data.var(axis=0, ddof=1)
    total_var = data.sum(axis=1).var(ddof=1)
    if total_var == 0:
        return None, n
    return (k / (k - 1)) * (1 - item_vars.sum() / total_var), n
 
def alpha_if_deleted(data):
    data = np.array(data, dtype=float)
    data = data[~np.isnan(data).any(axis=1)]
    k = data.shape[1]
    results = []
    for i in range(k):
        reduced = np.delete(data, i, axis=1)
        k2 = reduced.shape[1]
        item_vars = reduced.var(axis=0, ddof=1)
        total_var = reduced.sum(axis=1).var(ddof=1)
        if total_var == 0:
            results.append(None)
        else:
            results.append((k2 / (k2 - 1)) * (1 - item_vars.sum() / total_var))
    return results

In [22]:
print("=" * 55)
print("CRONBACH'S ALPHA")
print("=" * 55)
 
blocks = [
    ("Awareness Scale (Block 2, 4 items — Q3 excluded)", awareness_cols),
    ("Trust Index (Block 3, 5 items)",                   trust_cols),
    ("Creepy Effect Scale (Block 4, 5 items)",           creepy_cols),
]
 
for name, cols in blocks:
    data = df[cols].values
    alpha, n = cronbach_alpha(data)
    if alpha is None:
        print(f"\n{name}: cannot be calculated")
        continue
    interp = (
        "Excellent (≥0.90)"    if alpha >= 0.90 else
        "Good (≥0.80)"         if alpha >= 0.80 else
        "Acceptable (≥0.70)"   if alpha >= 0.70 else
        "Questionable (≥0.60)" if alpha >= 0.60 else
        "Poor (<0.60)"
    )
    print(f"\n{name}:")
    print(f"  N = {n}, k = {len(cols)}")
    print(f"  Cronbach's α = {alpha:.3f}  [{interp}]")
 
    aids = alpha_if_deleted(data)
    print(f"  Alpha if item deleted:")
    for i, (col, a) in enumerate(zip(cols, aids)):
        marker = " ← consider removing" if (a is not None and a > alpha + 0.05) else ""
        print(f"    Q{i+1}: α = {a:.3f}{marker}")

CRONBACH'S ALPHA

Awareness Scale (Block 2, 4 items — Q3 excluded):
  N = 123, k = 4
  Cronbach's α = 0.804  [Good (≥0.80)]
  Alpha if item deleted:
    Q1: α = 0.680
    Q2: α = 0.695
    Q3: α = 0.688
    Q4: α = 0.916 ← consider removing

Trust Index (Block 3, 5 items):
  N = 123, k = 5
  Cronbach's α = 0.739  [Acceptable (≥0.70)]
  Alpha if item deleted:
    Q1: α = 0.648
    Q2: α = 0.670
    Q3: α = 0.772
    Q4: α = 0.587
    Q5: α = 0.759

Creepy Effect Scale (Block 4, 5 items):
  N = 123, k = 5
  Cronbach's α = 0.965  [Excellent (≥0.90)]
  Alpha if item deleted:
    Q1: α = 0.950
    Q2: α = 0.954
    Q3: α = 0.949
    Q4: α = 0.951
    Q5: α = 0.975


In [23]:
# ================================================================
# SPEARMAN'S RANK CORRELATION
# ================================================================
print("\n" + "=" * 55)
print("SPEARMAN'S RANK CORRELATION")
print("=" * 55)
 
df['awareness_score'] = df[awareness_cols].sum(axis=1)
df['trust_score']     = df[trust_cols].sum(axis=1)
df['creepy_score']    = df[creepy_cols].sum(axis=1)
 
pairs = [
    ("Awareness → Trust",         'awareness_score', 'trust_score'),
    ("Awareness → Creepy Effect", 'awareness_score', 'creepy_score'),
    ("Trust → Creepy Effect",     'trust_score',     'creepy_score'),
]
 
for label, x, y in pairs:
    mask = df[x].notna() & df[y].notna()
    r, p = stats.spearmanr(df.loc[mask, x], df.loc[mask, y])
    sig = "p < 0.001 ✓✓✓" if p < 0.001 else "p < 0.01 ✓✓" if p < 0.01 else "p < 0.05 ✓" if p < 0.05 else "not significant"
    strength = (
        "strong"   if abs(r) >= 0.70 else
        "moderate" if abs(r) >= 0.40 else
        "weak"
    )
    direction = "positive" if r > 0 else "negative"
    print(f"\n{label}:")
    print(f"  r = {r:.3f},  p = {p:.4f}  ({sig})")
    print(f"  → {strength} {direction} relationship")


SPEARMAN'S RANK CORRELATION

Awareness → Trust:
  r = -0.482,  p = 0.0000  (p < 0.001 ✓✓✓)
  → moderate negative relationship

Awareness → Creepy Effect:
  r = 0.490,  p = 0.0000  (p < 0.001 ✓✓✓)
  → moderate positive relationship

Trust → Creepy Effect:
  r = -0.741,  p = 0.0000  (p < 0.001 ✓✓✓)
  → strong negative relationship


In [24]:
# ================================================================
# DESCRIPTIVE STATISTICS
# ================================================================
print("\n" + "=" * 55)
print("DESCRIPTIVE STATISTICS (composite scores)")
print("=" * 55)
 
score_info = [
    ("Awareness (4 items, max=20)", 'awareness_score', 4),
    ("Trust Index (5 items, max=25)", 'trust_score', 5),
    ("Creepy Effect (5 items, max=25)", 'creepy_score', 5),
]
 
for label, col, k in score_info:
    s = df[col].dropna()
    max_score = k * 5
    mean_per_item = s.mean() / k
    median_per_item = s.median() / k
    sd_per_item = s.std() / k
    pct = (s.mean() / max_score) * 100
    print(f"\n{label}:")
    print(f"  Composite: Mean = {s.mean():.2f}/{max_score}, SD = {s.std():.2f}, Min = {s.min():.0f}, Max = {s.max():.0f}")
    print(f"  Per-item (1-5 scale): Mean = {mean_per_item:.2f}, Median = {median_per_item:.2f}, SD = {sd_per_item:.2f}")
    print(f"  % of maximum: {pct:.1f}%")


DESCRIPTIVE STATISTICS (composite scores)

Awareness (4 items, max=20):
  Composite: Mean = 15.52/20, SD = 3.52, Min = 4, Max = 20
  Per-item (1-5 scale): Mean = 3.88, Median = 4.25, SD = 0.88
  % of maximum: 77.6%

Trust Index (5 items, max=25):
  Composite: Mean = 11.89/25, SD = 4.04, Min = 5, Max = 24
  Per-item (1-5 scale): Mean = 2.38, Median = 2.40, SD = 0.81
  % of maximum: 47.6%

Creepy Effect (5 items, max=25):
  Composite: Mean = 19.46/25, SD = 4.53, Min = 5, Max = 25
  Per-item (1-5 scale): Mean = 3.89, Median = 4.20, SD = 0.91
  % of maximum: 77.9%
